 ## EDA

In [1]:
import pandas as pd

df = pd.read_excel('dataset/Online_Retail.xlsx')
df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## Basic stats

Line-item transactions from a UK-based online gift retailer (UCI Online Retail). Period: 1 Dec 2010 – 9 Dec 2011.

In [4]:
overview = pd.DataFrame({
    "value": [
        df.shape[0],
        df.shape[1],
        df["InvoiceNo"].nunique(),
        df["StockCode"].nunique(),
        df["CustomerID"].nunique(),
        df["Country"].nunique(),
        df["InvoiceDate"].min(),
        df["InvoiceDate"].max(),
        (df["Quantity"] * df["UnitPrice"]).sum(),
    ]
}, index=[
    "rows",
    "columns",
    "unique invoices",
    "unique stock codes",
    "unique customers (non-null)",
    "countries",
    "first invoice",
    "last invoice",
    "line revenue (qty * unit price)",
])

display(overview)
print("\nColumn types")
display(df.dtypes.to_frame("dtype"))
print("\nNumeric summary")
display(df.describe(include="all"))
print("\nTop countries by line count")
display(df["Country"].value_counts().head(10).to_frame("lines"))

,value
rows,541909
columns,8
unique invoices,25900
unique stock codes,4070
unique customers (non-null),4372
countries,38
first invoice,2010-12-01 08:26:00
last invoice,2011-12-09 12:50:00
line revenue (qty * unit price),9747747.934



Column types


,dtype
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
UnitPrice,float64
CustomerID,float64
Country,str



Numeric summary


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909.0,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900.0,4070,4223,NaN,NaN,NaN,NaN,38
top,573585.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1114.0,2313,2369,NaN,NaN,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570,NaN
min,NaN,NaN,NaN,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,2011-03-28 11:34:00,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,2011-07-19 17:17:00,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,2011-10-19 11:27:00,4.130000,16791.000000,NaN
max,NaN,NaN,NaN,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000,NaN



Top countries by line count


,lines
Country,
United Kingdom,495478
Germany,9495
France,8557
EIRE,8196
Spain,2533
Netherlands,2371
Belgium,2069
Switzerland,2002
Portugal,1519


## Data quality checks

In [6]:
inv = df["InvoiceNo"].astype(str)

quality = pd.DataFrame({
    "count": [
        df.isna().sum()["Description"],
        df.isna().sum()["CustomerID"],
        df.duplicated().sum(),
        (df["Quantity"] < 0).sum(),
        (df["UnitPrice"] < 0).sum(),
        (df["UnitPrice"] == 0).sum(),
        inv.str.startswith("C").sum(),
        ((df["Quantity"] < 0) & ~inv.str.startswith("C")).sum(),
        inv.str.startswith("A").sum(),
    ]
}, index=[
    "missing Description",
    "missing CustomerID",
    "exact duplicate rows",
    "negative Quantity",
    "negative UnitPrice",
    "zero UnitPrice",
    "cancellation invoices (InvoiceNo starts with C)",
    "negative Quantity without C prefix",
    "bad-debt adjustments (InvoiceNo starts with A)",
])
quality["pct of rows"] = (quality["count"] / len(df) * 100).round(2)
display(quality)

print("Missing values by column")
miss = df.isna().sum().to_frame("missing")
miss["pct"] = (miss["missing"] / len(df) * 100).round(2)
display(miss)

print("Negative unit price rows (adjustments)")
display(df.loc[df["UnitPrice"] < 0, ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID"]])

print("Stock codes with more than one Description")
desc_variants = (
    df.dropna(subset=["Description"])
      .groupby("StockCode")["Description"]
      .nunique()
)
print(int((desc_variants > 1).sum()), "stock codes")

,count,pct of rows
missing Description,1454,0.27
missing CustomerID,135080,24.93
exact duplicate rows,5268,0.97
negative Quantity,10624,1.96
negative UnitPrice,2,0.00
zero UnitPrice,2515,0.46
cancellation invoices (InvoiceNo starts with C),9288,1.71
negative Quantity without C prefix,1336,0.25
bad-debt adjustments (InvoiceNo starts with A),3,0.00


Missing values by column


,missing,pct
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


Negative unit price rows (adjustments)


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
299983,A563186,B,Adjust bad debt,1,-11062.06,NaN
299984,A563187,B,Adjust bad debt,1,-11062.06,NaN


Stock codes with more than one Description
650 stock codes


### Immediate observations

- **Missing customers:** `CustomerID` is null on **135,080 rows (24.9%)**. Guest / unregistered checkouts cannot be used in customer-level RFM without dropping or imputing.
- **Missing product names:** `Description` is null on **1,454 rows (0.27%)**. All of those also have `UnitPrice == 0`, so they look like incomplete stock movements, not sales.
- **Duplicates:** **5,268 exact duplicate rows**. Likely repeated line items or load artifacts; decide whether to drop before aggregating.
- **Cancellations / returns:** **10,624 negative quantities**. Most (**9,288**) are invoices starting with `C`. **1,336** negative-qty rows do **not** start with `C` — warehouse adjustments, not clean returns.
- **Invalid prices:** **2 rows** with `UnitPrice = -11062.06` (`Adjust bad debt`, invoice prefix `A`). **2,515 rows** with `UnitPrice = 0`, including notes like `check`, `damages`, `?` mixed in with real product names.
- **Non-product stock codes:** codes such as `POST` (postage, 1,256 lines) are not merchandise SKUs.
- **Inconsistent descriptions:** **650** stock codes have more than one `Description` (whitespace/case plus genuine label changes). **~113k** descriptions have leading/trailing spaces.
- **Types:** `CustomerID` is float because of NaNs; invoice IDs are mixed numeric/`C`/`A` strings. `Quantity` and `UnitPrice` extremes (±80,995 qty, unit price up to 38,970) will dominate sums if left in.
- **Geography:** **38 countries**, but **~91% of lines are United Kingdom** — country splits besides UK are small samples.

For most sales analysis, filter to `Quantity > 0`, `UnitPrice > 0`, drop exact duplicates, and treat missing `CustomerID` as a separate guest segment.

In [7]:
# --- snapshot before cleaning ---
def _snapshot(frame):
    inv = frame["InvoiceNo"].astype(str)
    return pd.Series({
        "rows": len(frame),
        "cols": frame.shape[1],
        "duplicate_rows": int(frame.duplicated().sum()),
        "missing_values": int(frame.isna().sum().sum()),
        "missing_Description": int(frame["Description"].isna().sum()),
        "missing_CustomerID": int(frame["CustomerID"].isna().sum()),
        "unique_invoices": frame["InvoiceNo"].nunique(),
        "unique_customers": frame["CustomerID"].nunique(),
        "cancellations_C": int(inv.str.startswith("C").sum()),
        "negative_qty": int((frame["Quantity"] < 0).sum()),
        "nonpositive_price": int((frame["UnitPrice"] <= 0).sum()),
        "revenue_sum": (frame["Quantity"] * frame["UnitPrice"]).sum(),
    })

before = _snapshot(df)

# work on a copy so the raw load stays intact
clean = df.copy()

# drop exact duplicate line items
clean = clean.drop_duplicates()

# drop rows missing product name or customer id
clean = clean.dropna(subset=["Description", "CustomerID"])

# cancelled / returned: InvoiceNo starting with C, or qty <= 0 (incl. non-C adjustments)
# also drop non-positive prices (free lines, damages notes, A* bad-debt adjustments)
inv = clean["InvoiceNo"].astype(str)
is_cancel_or_return = inv.str.startswith("C") | (clean["Quantity"] <= 0)
is_invalid_price = clean["UnitPrice"] <= 0
clean = clean.loc[~is_cancel_or_return & ~is_invalid_price].copy()

# line revenue for remaining sales
clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]

# CustomerID is whole numbers once NaNs are gone
clean["CustomerID"] = clean["CustomerID"].astype(int)

after = _snapshot(clean)

print("BEFORE")
print(before.to_string())
print("\nAFTER")
print(after.to_string())
print("\nDELTA (after - before)")
print((after - before).to_string())

clean.head()

BEFORE
rows                    541909.000
cols                         8.000
duplicate_rows            5268.000
missing_values          136534.000
missing_Description       1454.000
missing_CustomerID      135080.000
unique_invoices          25900.000
unique_customers          4372.000
cancellations_C           9288.000
negative_qty             10624.000
nonpositive_price         2517.000
revenue_sum            9747747.934

AFTER
rows                    392692.000
cols                         9.000
duplicate_rows               0.000
missing_values               0.000
missing_Description          0.000
missing_CustomerID           0.000
unique_invoices          18532.000
unique_customers          4338.000
cancellations_C              0.000
negative_qty                 0.000
nonpositive_price            0.000
revenue_sum            8887208.894

DELTA (after - before)
rows                  -149217.00
cols                        1.00
duplicate_rows          -5268.00
missing_values        -

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [ ]:
import matplotlib.pyplot as plt

# invoice-level grain (one row = one order)
orders = (
    clean.groupby("InvoiceNo", as_index=False)
         .agg(
             InvoiceDate=("InvoiceDate", "min"),
             Country=("Country", "first"),
             Quantity=("Quantity", "sum"),
             Revenue=("Revenue", "sum"),
         )
)
orders["YearMonth"] = orders["InvoiceDate"].dt.to_period("M").astype(str)
orders["AOV"] = orders["Revenue"]  # order value; AOV = mean of this

# =============================================================================
# 1) Sales & revenue trends over time
# =============================================================================
monthly = (
    orders.groupby("YearMonth", as_index=False)
          .agg(
              invoices=("InvoiceNo", "nunique"),
              quantity=("Quantity", "sum"),
              revenue=("Revenue", "sum"),
              aov=("Revenue", "mean"),
          )
)
print("MONTHLY TREND")
display(monthly.round(2))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(monthly["YearMonth"], monthly["revenue"], marker="o")
axes[0].set_ylabel("Revenue")
axes[0].set_title("Revenue by month")
axes[0].tick_params(axis="x", rotation=45)
axes[1].plot(monthly["YearMonth"], monthly["invoices"], marker="o", color="tab:orange")
axes[1].set_ylabel("Invoices")
axes[1].set_title("Orders by month")
axes[1].tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

# =============================================================================
# 2) Geographic distribution
# =============================================================================
geo = (
    orders.groupby("Country", as_index=False)
          .agg(
              invoices=("InvoiceNo", "nunique"),
              quantity=("Quantity", "sum"),
              revenue=("Revenue", "sum"),
              aov=("Revenue", "mean"),
          )
          .sort_values("revenue", ascending=False)
)
geo["revenue_share_pct"] = (geo["revenue"] / geo["revenue"].sum() * 100).round(2)
print("GEOGRAPHY (all countries, by revenue)")
display(geo.round(2))

top_geo = geo.head(10)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh(top_geo["Country"][::-1], top_geo["revenue"][::-1])
axes[0].set_xlabel("Revenue")
axes[0].set_title("Top 10 countries by revenue")
axes[1].barh(top_geo["Country"][::-1], top_geo["aov"][::-1], color="tab:green")
axes[1].set_xlabel("AOV")
axes[1].set_title("AOV in top 10 revenue countries")
fig.tight_layout()
plt.show()

# =============================================================================
# 3) Top performing products
# =============================================================================
products = (
    clean.groupby(["StockCode", "Description"], as_index=False)
         .agg(
             lines=("InvoiceNo", "count"),
             invoices=("InvoiceNo", "nunique"),
             quantity=("Quantity", "sum"),
             revenue=("Revenue", "sum"),
         )
)
print("TOP 15 PRODUCTS BY REVENUE")
display(products.nlargest(15, "revenue").round(2))
print("TOP 15 PRODUCTS BY QUANTITY")
display(products.nlargest(15, "quantity").round(2))

top_rev = products.nlargest(10, "revenue")
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_rev["Description"][::-1], top_rev["revenue"][::-1], color="tab:purple")
ax.set_xlabel("Revenue")
ax.set_title("Top 10 products by revenue")
fig.tight_layout()
plt.show()

# =============================================================================
# 4) Average order value
# =============================================================================
aov_overall = pd.Series({
    "orders": len(orders),
    "total_revenue": orders["Revenue"].sum(),
    "aov_mean": orders["Revenue"].mean(),
    "aov_median": orders["Revenue"].median(),
    "items_per_order_mean": orders["Quantity"].mean(),
})
print("AOV OVERALL")
print(aov_overall.round(2).to_string())

print("\nAOV BY MONTH")
display(monthly[["YearMonth", "invoices", "revenue", "aov"]].round(2))

print("AOV BY COUNTRY (top 15 by revenue)")
display(geo.head(15)[["Country", "invoices", "revenue", "aov"]].round(2))
